In [ ]:
# Grounding DINO
from pathlib import Path

import torch
import matplotlib.pyplot as plt

from groundingdino.util.inference import load_model, load_image, predict, annotate

# Resolve paths relative to this notebook directory
ROOT = Path.cwd().parent / "GroundingDINO"
CONFIG_PATH = ROOT / "groundingdino/config/GroundingDINO_SwinB_cfg.py"
WEIGHTS_PATH = ROOT / "weights/groundingdino_swinb_cogcoor.pth"
IMAGE_PATH = ROOT / ".asset/cat_dog.jpeg"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = load_model(str(CONFIG_PATH), str(WEIGHTS_PATH), device=device)
TEXT_PROMPT = "cat . dog ."
BOX_THRESHOLD = 0.35
TEXT_THRESHOLD = 0.25

image_source, image = load_image(str(IMAGE_PATH))

boxes, logits, phrases = predict(
    model=model,
    image=image,
    caption=TEXT_PROMPT,
    box_threshold=BOX_THRESHOLD,
    text_threshold=TEXT_THRESHOLD,
)
print(f"boxes: {boxes}, logits: {logits}, phrases: {phrases}")
print(f"boxes_shape: {boxes.shape}, logits_shape: {logits.shape}, phrases_length: {len(phrases)}")

annotated_frame = annotate(
    image_source=image_source,
    boxes=boxes,
    logits=logits,
    phrases=phrases,
 )

plt.figure(figsize=(10, 10))
plt.imshow(annotated_frame)
plt.axis('off')
plt.show()

In [ ]:
# SAM2
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Resolve paths relative to this notebook directory
ROOT = Path.cwd().parent / "sam2"
CHECKPOINT = ROOT / "checkpoints/sam2.1_hiera_large.pt"
MODEL_CONFIG = "configs/sam2.1/sam2.1_hiera_l.yaml"
IMAGE_PATH = ROOT / "notebooks/images/truck.jpg"
device = "cuda" if torch.cuda.is_available() else "cpu"
autocast_context = (
    torch.autocast("cuda", dtype=torch.bfloat16)
    if device == "cuda"
    else nullcontext()
)

# Build the SAM2 model and create a predictor
model = build_sam2(MODEL_CONFIG, str(CHECKPOINT), device=device)
predictor = SAM2ImagePredictor(model)
print("Model loaded")
# Load the input image
image = Image.open(IMAGE_PATH).convert("RGB")

# Display the sorted masks, scores, and input points/boxes
def show_masks(image, masks, scores, 
               input_points=None, input_labels=None, input_box=None,
               max_shown=3, fig_title=None):
    if (input_points is None) ^ (input_labels is None):
        raise ValueError("Both input_labels and input_points should be provided")
    if input_points is None:
        input_points = []
    if input_labels is None:
        input_labels = []
    if input_box is None:
        input_box = []
    # Sort masks by scores in descending order
    sorted_ind = np.argsort(scores)[::-1]
    masks = masks[sorted_ind]
    scores = scores[sorted_ind]
    num_masks_to_show = min(max_shown, len(sorted_ind))

    # Display each mask with the corresponding score and input points/boxes
    fig, axes = plt.subplots(1, num_masks_to_show, figsize=(num_masks_to_show * 5, 5))
    if num_masks_to_show == 1:
        axes = [axes]
    for i in range(num_masks_to_show):
        mask_overlay = np.zeros((*masks[i].shape, 4), dtype=np.float32)
        mask_overlay[masks[i] > 0] = [30 / 255, 144 / 255, 1.0, 0.5]
        axes[i].imshow(image)
        axes[i].imshow(mask_overlay)
        for input_point, input_label in zip(input_points, input_labels):
            axes[i].scatter(
                input_point[0],
                input_point[1],
                color="yellow" if input_label == 1 else "red",
                marker="*" if input_label == 1 else "o",
                s=250,
                edgecolor="black",
            )
        if len(input_box) == 4:
            rect = plt.Rectangle(
                (input_box[0], input_box[1]),
                input_box[2] - input_box[0],
                input_box[3] - input_box[1],
                linewidth=2,
                edgecolor="green",
                facecolor="none",
            )
            axes[i].add_patch(rect)
        axes[i].axis("off")
        axes[i].set_title(f"Mask {i+1}, Score: {scores[i]:.4f}")
        
    if fig_title is not None:
        fig.suptitle(fig_title)
        plt.tight_layout()
    plt.show()

### Inference with one positive point input
# Prepare the input point and label
input_points = np.array([[image.width / 2, image.height / 2]])
input_labels = np.array([1])
# Inference
with torch.inference_mode(), autocast_context:
    predictor.set_image(image)
    masks, scores, logits = predictor.predict(
        point_coords=input_points,
        point_labels=input_labels,
        multimask_output=True,
    )
# Show the result
print(f"masks: {masks}, scores: {scores}, logits: {logits}")
print(f"masks_shape: {masks.shape}, scores_shape: {scores.shape}, logits_shape: {logits.shape}")

show_masks(image, masks, scores, input_points=input_points, input_labels=input_labels)

In [ ]:
# Depth Anything 3
import glob
import os
from pathlib import Path

import torch

from depth_anything_3.api import DepthAnything3

ROOT = Path.cwd().parent / "Depth-Anything-3"
MODEL_NAME = "DA3MONO-LARGE"
EXAMPLE_PATH = ROOT / "assets/examples/SOH"

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DepthAnything3.from_pretrained(f"depth-anything/{MODEL_NAME}")
model = model.to(device=device)

images = sorted(glob.glob(str(EXAMPLE_PATH / "*.png")))

# Multiple images inference
prediction = model.inference(
    images,  # List of image paths, PIL images or numpy arrays
)
# prediction.processed_images : [N, H, W, 3] uint8   array
print(prediction.processed_images.shape)
# prediction.depth            : [N, H, W]    float32 array
print(prediction.depth.shape)

# Metric and Monocular models do not output camera parameters
if MODEL_NAME not in ["DA3METRIC-LARGE", "DA3MONO-LARGE"]:
    # prediction.conf             : [N, H, W]    float32 array
    print(prediction.conf.shape)  
    # prediction.extrinsics       : [N, 3, 4]    float32 array # opencv w2c or colmap format
    print(prediction.extrinsics.shape)
    # prediction.intrinsics       : [N, 3, 3]    float32 array
    print(prediction.intrinsics.shape)